In [ ]:
import numpy as np

try:
    import cupy as cp
except Exception:
    cp = None

from k2tree import K2Tree


In [ ]:
USE_GPU = cp is not None
K = 2
S_WORDS = 1024


In [ ]:
from topologies import square_torus

adj = square_torus(100)
N = 100 ** 2

k2 = K2Tree.from_adjdict(
    adj,
    N=N,
    k=K,
    s_words=S_WORDS,
    use_gpu=USE_GPU,
    verify_all=False,
)

# random query batch + some fixed ones
rng = np.random.default_rng(0)
nq = 10000
u = rng.integers(0, N, size=nq, dtype=np.int32)
v = rng.integers(0, N, size=nq, dtype=np.int32)

cpu = np.fromiter((1 if int(vv) in adj.get(int(uu), []) else 0 for uu, vv in zip(u, v)),
                  count=nq, dtype=np.uint8)
gpu = k2.adj_batch(u, v)

mismatch = np.nonzero(cpu != gpu)[0]
print("K =", K, "N =", k2.N, "H =", k2.H, "Npad =", k2.Npad, "T_nbits =", k2.T_nbits)
print("T_words =", int(k2.T_words.size), "L_words =", int(k2.L_words.size))
print("mismatches:", int(mismatch.size))

if mismatch.size:
    i = int(mismatch[0])
    print("first mismatch at i =", i, "u,v =", int(u[i]), int(v[i]), "cpu=", int(cpu[i]), "gpu=", int(gpu[i]))
    raise AssertionError("k²-tree adjacency mismatch")
else:
    print("OK: CPU dict adjacency matches k²-tree adjacency on test batch.")


In [ ]:
# ---------------- k^2-tree navigation primitives (compile-time K) ----------------
# Set this once for max speed (compile-time constants in CUDA):
K = 5  # <-- change here
S_WORDS = 1024  # rank superblock size in 32-bit words (tune later if needed)


Verification note

`verify_all=True` does a full O(N^2) adjacency check. For large N, use `verify_all=False`
or lower N during development. Progress prints every `verify_progress_every` rows.


In [ ]:
# Optional: inspect a mismatch with a trace
if mismatch.size:
    i = int(mismatch[0])
    print("trace u,v =", int(u[i]), int(v[i]))
    _ = k2.trace(int(u[i]), int(v[i]))


In [ ]:
cpu[3686]

In [ ]:
u[3686]

In [ ]:
v[3686]

Debug helpers

`k2.trace(u, v)` prints a traversal path for a single query.


In [ ]:
# Example trace on a random pair
uu = int(u[0])
vv = int(v[0])
_ = k2.trace(uu, vv)
print("dict says:", 1 if vv in adj.get(uu, []) else 0)


In [ ]:
# Force kernel recompile on GPU (no restart needed)
if k2.on_gpu:
    k2.recompile_kernel()
